In [ ]:
import numpy as np
from pathlib import Path
from labdata.schema import *
from labdata import chronic_paper as cp
import matplotlib.pyplot as plt
import seaborn as sns
from one.api import ONE
import h5py
from tqdm import tqdm
from spks.waveforms import compute_waveform_metrics, waveforms_position
from spks.viz import plot_drift_raster 
import matplotlib

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

one = ONE()

%load_ext autoreload
%autoreload 2
#%matplotlib qt
#%matplotlib widget


In [ ]:
# target dates for chronic timepoint comparison
from datetime import timedelta
COMPARISON_TIMEPOINTS = [timedelta(days=7), timedelta(days=28), timedelta(days=49), timedelta(days=70)]
SORT_PARAM = 5
CRITERIA_PARAM = 7 # was 1
NPRE = 45 # samples before peak of waveform

SAVEFIGS = False
FIGPATH = Path(r'C:\Data\churchland\chronic_manuscript_figs\raw_plots\drift_metric_supps')

In [ ]:
chronic_insertions = cp.IBLMatchedInsertion()
chronic_recording_keys = []
for ins in chronic_insertions:
    recordings = (EphysRecording() & ins) * Session()

    for timepoint in COMPARISON_TIMEPOINTS:
        target_datetime = ins['procedure_datetime'] + timepoint
        recordings_with_time_diff = recordings.proj(days_from_target='ABS(TIMESTAMPDIFF(DAY, session_datetime, "{}"))'.format(target_datetime.strftime('%Y-%m-%d %H:%M:%S')))
        key = recordings_with_time_diff.fetch(order_by='days_from_target ASC', as_dict=True, limit=1)[0]
        if key['days_from_target'] > 3:
            print(f"{key['days_from_target']} away from target date {timepoint}")
        chronic_recording_keys.append(key)
chronic_probe_recordings = EphysRecording.proj() & chronic_recording_keys
acute_probe_recordings = EphysRecording.proj() & cp.IBLMatchedInsertion().to_ephys_session()

In [ ]:
# launch spike sorting for all recordings
all_probe_sessions =  (EphysRecording() & acute_probe_recordings.fetch('KEY') + chronic_probe_recordings.fetch('KEY')) - (SpikeSorting & dict(parameter_set_num=SORT_PARAM))
labdata_submission_commands = []
t = []
for k in all_probe_sessions.fetch(order_by='subject_name, session_name'):
    dd = f'labdata2 run spks -t aws -a {k["subject_name"]} -s {k["session_name"]} --force-submit -- -m ks4.0'
    t.append(dict(session_name=k['session_name'],
                   subject_name=k['subject_name'],))
    #dd = f'labdata2 run detect -t aws -a {k["subject_name"]} -s {k["session_name"]}'
    labdata_submission_commands.append(dd)
    #os.system(dd) # uncomment to run
labdata_submission_commands = np.unique(labdata_submission_commands)
print(f'There are {len(labdata_submission_commands)} sessions to process.')
print('\n'.join(labdata_submission_commands))

all_probe_session_keys = all_probe_sessions.fetch('KEY')

In [ ]:
# get the probe keys so we can run dredge on each probe-session
acute_session_probe_keys = EphysRecording.ProbeSetting() & (cp.IBLMatchedInsertion().EphysRecording().proj('session_name', 
                                                                                                           'dataset_name',
                                                                                                           chronic_mouse='subject_name',
                                                                                                           chronic_probe_id='probe_id',
                                                                                                           subject_name='matched_subject_name',
                                                                                                           probe_num='matched_probe_num'))
chronic_session_probe_keys = (EphysRecording.ProbeSetting() & chronic_probe_recordings)
chronic_probes = dj.U('subject_name', 'probe_num') & chronic_session_probe_keys # sessions have the same trajectories 

In [ ]:
def pid2trajectory(pid):
    trajectory = one.alyx.rest('trajectories', 'list', 
                                 probe_insertion=pid,
                                 provenance='Ephys aligned histology track')
    assert len(trajectory) == 1, f"Found {len(trajectory)} trajectories for probe {chronic_pid}"
    trajectory = trajectory[0]
    mlap = [trajectory['x'], trajectory['y']]
    angles = [90-trajectory['theta'], trajectory['roll'], trajectory['phi']-90] # convert to vvasp coordinate system
    return mlap, angles, trajectory['depth']

In [ ]:
# make a vvasp plot of the insertions
from tqdm import tqdm
from brainbox.io.one import SpikeSortingLoader
from vvasp.viz_objects import Probe
from vvasp.atlas import VVASPAtlas
from pyvistaqt import BackgroundPlotter
clip_height = 100

plotter = BackgroundPlotter()

atlas = VVASPAtlas(plotter, mapping='Beryl')
regions = ['MOp','MOs','ACAd','ACAv','VISp','VISpm','VISam','VISa']
for r in regions:
    atlas.add_atlas_region_mesh(r,'left')

vvasp_probes = []
for i,c in enumerate(tqdm(chronic_probes)):
    chronic_insertion = ProbeInsertion * Session * UnitCount * (chronic_session_probe_keys & c) & dict(unit_criteria_id=CRITERIA_PARAM,
                                                                                                       parameter_set_num=SORT_PARAM)
    chronic_insertion = chronic_insertion.proj('sua','mua', days_from_insertion='DATEDIFF(session_datetime, procedure_datetime)')
    chronic_session = (chronic_insertion * Session).fetch('subject_name','session_name', limit=1,as_dict=True)[0]
    chronic_eid = one.path2eid(f"{chronic_session['subject_name']}/{chronic_session['session_name']}")
    pids, probes = one.eid2pid(chronic_eid)
    # select the proper pid, where probe contains c['probe_num']
    chronic_pid = [pid for pid, probe in zip(pids, probes) if str(c['probe_num']) in probe][0]
    mlap, angles, depth = pid2trajectory(chronic_pid)
    p = Probe('NP1',plotter, root_intersection_mesh=atlas.meshes['root'])
    p.drive_probe_from_entry(mlap, angles, depth)
    p.make_inactive()
    for mesh,actor in zip(p.meshes, p.actors):
        #clip_height = p.depth + 100
        mesh.clip(normal=(0,0,-1), origin=(0, 0, clip_height), invert=False, inplace=True)
        actor.prop.color = 'black'
    p.ball_actor.prop.color = 'black'
    vvasp_probes.append(p)

    #ssl = SpikeSortingLoader(pid=chronic_pid, one=one)
    #channels = ssl.load_channels()
    #xyz = np.stack([channels['x'], channels['y'], channels['z']], axis=1) * 1e6
    #plotter.add_points(xyz,
    #               color='black',
    #               point_size=5,
    #               render_points_as_spheres=True)
    #clusters = ssl.merge_clusters(spikes, clusters, channels)

    # now do acute data
    matched_acute = (cp.IBLMatchedInsertion().EphysRecording * (cp.IBLMatchedInsertion().EphysRecording.proj() & chronic_insertion)).proj('session_name',ppp='probe_id', temp='subject_name',subject_name='matched_subject_name', probe_num='matched_probe_num')
    for m in matched_acute:
        # strip leading underscore from subject name
        subject_name = m['subject_name'].lstrip('_')
        acute_eid = one.path2eid(f"{subject_name}/{m['session_name']}")
        # get the pid as above using m['probe_num]
        pids, probes = one.eid2pid(acute_eid)
        acute_pid = [pid for pid, probe in zip(pids, probes) if str(m['probe_num']) in probe][0]
        acute_pname = [probe for pid, probe in zip(pids, probes) if str(m['probe_num']) in probe][0]
        mlap, angles, depth = pid2trajectory(acute_pid)
        p = Probe('NP1',plotter, root_intersection_mesh=atlas.meshes['root'])
        p.drive_probe_from_entry(mlap, angles, depth)
        for mesh, actor in zip(p.meshes, p.actors):
            #clip_height = p.depth + 100
            mesh.clip(normal=(0,0,-1), origin=(0, 0, clip_height), invert=False, inplace=True)
            actor.prop.color = 'red'
        p.ball_actor.prop.color = 'red'
        vvasp_probes.append(p)
    #break

        #ssl = SpikeSortingLoader(pid=acute_pid, one=one)
        #channels = ssl.load_channels()
        #xyz = np.stack([channels['x'], channels['y'], channels['z']], axis=1) * 1e6
        #plotter.add_points(xyz,
        #           color='red',
        #           point_size=5,
        #           render_points_as_spheres=True)

In [ ]:
#UnitCount.populate()
plt.figure()
ACUTE_DAYS_OFFSET = -10
#colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
colors = sns.color_palette("hls", len(chronic_probes))
fig, axs = plt.subplots(2,1, figsize=(10,10))

stats_rows = []
for i,c in enumerate(chronic_probes):
    chronic_insertion = ProbeInsertion * Session * UnitCount * (chronic_session_probe_keys & c) & dict(unit_criteria_id=CRITERIA_PARAM,
                                                                                                       parameter_set_num=SORT_PARAM)
    chronic_insertion = chronic_insertion.proj('sua','mua', days_from_insertion='DATEDIFF(session_datetime, procedure_datetime)')
    days, sua, mua = chronic_insertion.fetch('days_from_insertion','sua','mua', order_by='days_from_insertion ASC')
    axs[0].plot(days, sua, marker='o', color=colors[i])
    axs[1].plot(days, mua, marker='o', color=colors[i])
    for j,(s,m) in enumerate(zip(sua,mua)):
        stats_rows.append((i, 1, j, 0, m, s))

    # now do acute data
    matched_acute = (cp.IBLMatchedInsertion().EphysRecording * (cp.IBLMatchedInsertion().EphysRecording.proj() & chronic_insertion)).proj('session_name',ppp='probe_id', temp='subject_name',subject_name='matched_subject_name', probe_num='matched_probe_num')
    temp = (cp.IBLMatchedInsertion() * (cp.IBLMatchedInsertion().EphysRecording.proj() & chronic_insertion))#.proj('session_name',ppp='probe_id', temp='subject_name',subject_name='matched_subject_name', probe_num='matched_probe_num')
    temp = temp.fetch('pid')
    matched_acute = matched_acute * UnitCount & dict(parameter_set_num=SORT_PARAM,unit_criteria_id=CRITERIA_PARAM)
    acute_sua, acute_mua = matched_acute.fetch('sua','mua')
    print(acute_sua)
    #print(temp)
    biggest_ind = np.argmax(acute_sua)
    ttt = matched_acute.fetch('subject_name','session_name','probe_num', as_dict=True)[biggest_ind]
    ttt = matched_acute.fetch('subject_name','session_name','probe_num', as_dict=True)
    print(ttt)
    offset = np.random.normal(scale=1, size=len(acute_sua)) + ACUTE_DAYS_OFFSET
    axs[0].scatter(offset, acute_sua, marker='o', color=colors[i], facecolors='none')
    axs[1].scatter(offset, acute_mua, marker='o', color=colors[i], facecolors='none')
    for j,(s,m) in enumerate(zip(acute_sua,acute_mua)):
        stats_rows.append((i, 0, 0, j, m, s))

xticks = [ACUTE_DAYS_OFFSET] + [c.days for c in COMPARISON_TIMEPOINTS]
xticklabels = ['Acute'] + [c.days for c in COMPARISON_TIMEPOINTS]
axs[0].set_xticks(xticks)
axs[0].set_xticklabels(xticklabels)
axs[1].set_xticks(xticks)
axs[1].set_xticklabels(xticklabels)
axs[0].set_ylabel('Num single units')
axs[1].set_ylabel('Num multi units')
axs[0].set_xlabel('Days post insertion')
axs[1].set_xlabel('Days post insertion')
if SAVEFIGS:
    plt.savefig(FIGPATH / 'acute_chronic_unit_count.pdf', dpi=800)

stats_table = pd.DataFrame(stats_rows, columns=['insertion_site','is_chronic','timepoint','session_num','multi_units','single_units'])
stats_table.to_csv(Path().resolve().parent / 'stats_tables' / 'acute_chronic_unit_count.csv', index=False)

In [ ]:
# first, just show the number of units lost in each recording for acute and chronic
acute_units = (SpikeSorting.Unit * acute_session_probe_keys) & dict(parameter_set_num=SORT_PARAM)
chronic_units = (SpikeSorting.Unit * chronic_session_probe_keys) & dict(parameter_set_num=SORT_PARAM)
acute_passes_first = acute_units.aggr(UnitCount.Unit & dict(unit_criteria_id=1, passes=1))
acute_passes_second = acute_units.aggr(UnitCount.Unit & dict(unit_criteria_id=7, passes=1))
acute_fails_second = acute_units.aggr(UnitCount.Unit & dict(unit_criteria_id=7, passes=0))
acute_fails_first = acute_units.aggr(UnitCount.Unit & dict(unit_criteria_id=1, passes=0))

chronic_passes_first = chronic_units.aggr(UnitCount.Unit & dict(unit_criteria_id=1, passes=1))
chronic_passes_second = chronic_units.aggr(UnitCount.Unit & dict(unit_criteria_id=7, passes=1))
chronic_fails_second = chronic_units.aggr(UnitCount.Unit & dict(unit_criteria_id=7, passes=0))
chronic_fails_first = chronic_units.aggr(UnitCount.Unit & dict(unit_criteria_id=1, passes=0))

acute_p1f2 = acute_passes_first * acute_fails_second
chronic_p1f2 = chronic_passes_first * chronic_fails_second

acute_files = AnalysisFile & (SpikeSorting & acute_p1f2).proj(file_path='waveforms_file')
chronic_files = AnalysisFile & (SpikeSorting & chronic_p1f2).proj(file_path='waveforms_file')

print(f'There are {len(acute_files)} and {len(chronic_files)} acute and chronic files to get.')
#acute_files.check_if_files_archived(suppress_error=True)
#chronic_files.check_if_files_archived(suppress_error=True)
#acute_files.get()
#chronic_files.get()


num_acute_first = EphysRecording.ProbeSetting.aggr(acute_passes_first, num_units='COUNT(*)').fetch('num_units')
num_acute_second = EphysRecording.ProbeSetting.aggr(acute_passes_second, num_units='COUNT(*)').fetch('num_units')

num_chronic_first = EphysRecording.ProbeSetting.aggr(chronic_passes_first, num_units='COUNT(*)').fetch('num_units')
num_chronic_second = EphysRecording.ProbeSetting.aggr(chronic_passes_second, num_units='COUNT(*)').fetch('num_units')

acute_y = np.stack([num_acute_first, num_acute_second])
acute_x = np.stack([np.zeros_like(num_acute_first), np.zeros_like(num_acute_second) + 1])
acute_x = acute_x + np.random.normal(scale=.01, size=acute_x.shape)

chronic_y = np.stack([num_chronic_first, num_chronic_second])
chronic_x = np.stack([np.zeros_like(num_chronic_first) + 3, np.zeros_like(num_chronic_second) + 4])
chronic_x = chronic_x + np.random.normal(scale=.01, size=chronic_x.shape)

delta_acute = num_acute_second - num_acute_first
delta_chronic = num_chronic_second - num_chronic_first

plt.figure()
plt.plot(acute_x, acute_y, color='k', alpha=.6)
plt.scatter(acute_x, acute_y, color='k')
plt.plot(chronic_x, chronic_y, color='k', alpha=.6)
plt.scatter(chronic_x, chronic_y, color='red')

plt.ylabel('Num single units')
plt.xticks([.5, 3.5], labels=['Acute sessions', 'Chronic sessions'])
if SAVEFIGS:
    plt.savefig(FIGPATH / 'metrics_compare_unit_count.pdf', dpi=800)

In [ ]:
# test if the number of units lost is different between acute and chronic
from scipy.stats import ranksums, wilcoxon
stat, p_value = ranksums(delta_acute, delta_chronic)
stat, p_value_acute = wilcoxon(num_acute_second.astype(int), num_acute_first.astype(int))
stat, p_value_chronic = wilcoxon(num_chronic_second.astype(int), num_chronic_first.astype(int))
p_value_acute, p_value_chronic, p_value
# paired rank sum test


In [ ]:
def load_waveforms(waveform_file, unit_ids, sampling_rate=30000, minutes=10, gain=None):
    with h5py.File(str(waveform_file), 'r') as f:
        early_waveforms, late_waveforms = [], []
        #for unit_id in unit_ids:
        for unit_id in tqdm(unit_ids, desc='Loading waveforms'):
            inds = f[str(unit_id)]['indices'][:]
            waveforms = (f[str(unit_id)]['waveforms'][:])
            if gain is not None:
                waveforms = waveforms* gain #* 0.06
            inds_in_minutes = inds / sampling_rate / 60
            early_inds = np.where(inds_in_minutes <= minutes)[0]
            late_inds = np.where(inds_in_minutes >= (inds_in_minutes[-1] - minutes))[0]
            #if len(early_inds) < 10 or len(late_inds) < 10:
            #    return np.array([]), np.array([])
            early_waveforms.append(np.nanmean(waveforms[early_inds], axis=0))
            late_waveforms.append(np.nanmean(waveforms[late_inds], axis=0))
    return np.array(early_waveforms), np.array(late_waveforms)

In [ ]:
def compute_early_and_late_waveform_metrics(unit_query):
    chan_positions, early_waveforms, late_waveforms = [], [], []
    all_early_metrics, all_late_metrics = [], []
    for counter, session in enumerate((SpikeSorting & unit_query).proj(file_path='waveforms_file')):
        if counter==2:
            break # TODO: remove me 
        units_from_session = unit_query & session
        unit_ids = units_from_session.fetch('unit_id')
        waveforms_file = (AnalysisFile & session).get()
        channel_coords, sampling_rate, gain = (SpikeSorting*EphysRecording.ProbeSetting*ProbeConfiguration & session).fetch1('sorting_channel_coords', 'sampling_rate','probe_gain')
        assert len(waveforms_file) == 1, "Expected exactly one waveform file per session."
        waveforms_file = waveforms_file[0]
        print(f'Running for acute session {session["subject_name"]} {session["session_name"]} with {len(unit_ids)} units.')

        early, late = load_waveforms(waveforms_file, unit_ids, sampling_rate=sampling_rate, gain=gain)
        early_waveforms.extend(early), late_waveforms.extend(late)
        _, peak_chans, _ = waveforms_position(early, channel_coords)
        early_metrics, late_metrics = [], []
        for i, peak_chan in enumerate(peak_chans):
            chan_positions.append(channel_coords)
            early_metrics.append(compute_waveform_metrics(early[i,:,peak_chan], NPRE, float(sampling_rate)))
            late_metrics.append(compute_waveform_metrics(late[i,:,peak_chan], NPRE, float(sampling_rate)))
        early_metrics, late_metrics = np.array(early_metrics), np.array(late_metrics)
        all_early_metrics.append(early_metrics)
        all_late_metrics.append(late_metrics)

    all_late_metrics = np.hstack(all_late_metrics)
    all_early_metrics = np.hstack(all_early_metrics)
    keys = all_late_metrics[0].keys()
    all_late_metrics = {k: np.hstack([m[k] for m in all_late_metrics]) for k in keys}
    all_early_metrics = {k: np.hstack([m[k] for m in all_early_metrics]) for k in keys}
    return all_early_metrics, all_late_metrics, chan_positions, early_waveforms, late_waveforms

#acute_early, acute_late,_,_,_ = compute_early_and_late_waveform_metrics(acute_p1f2)
#chronic_early, chronic_late,p,e,l = compute_early_and_late_waveform_metrics(chronic_p1f2)
#acute_early, acute_late,_,_,_ = compute_early_and_late_waveform_metrics(acute_passes_first)
#chronic_early, chronic_late,_,_,_ = compute_early_and_late_waveform_metrics(chronic_passes_first)

In [ ]:
#beginning_amps = np.abs(acute_early['peak_amplitude']) + np.abs(acute_early['trough_amplitude'])
#late_amps = np.abs(acute_late['peak_amplitude']) + np.abs(acute_late['trough_amplitude'])
#acute_abs_amp_drift = np.abs(beginning_amps - late_amps) / np.sum([beginning_amps, late_amps], axis=0)
#
#beginning_amps = np.abs(chronic_early['peak_amplitude']) + np.abs(chronic_early['trough_amplitude'])
#late_amps = np.abs(chronic_late['peak_amplitude']) + np.abs(chronic_late['trough_amplitude'])
#chronic_abs_amp_drift = np.abs(beginning_amps - late_amps) / np.sum([beginning_amps, late_amps], axis=0)
#
#x = np.random.normal(scale=.01, size=len(acute_abs_amp_drift))
#x2 = np.random.normal(loc=1, scale=.01, size=len(chronic_abs_amp_drift))
#
#plt.figure()
##plt.scatter(x, acute_abs_amp_drift)
##plt.scatter(x2, chronic_abs_amp_drift)
#bins = np.linspace(-10, 5, 40)
## convert to density
#
##plt.hist(acute_abs_amp_drift, bins=bins, alpha=.5, label='Acute')
##plt.hist(chronic_abs_amp_drift, bins=bins, alpha=.5, label='Chronic')
#plt.hist(np.log(acute_abs_amp_drift), bins=bins, alpha=.5, label='Acute', density=True)
#plt.hist(np.log(chronic_abs_amp_drift), bins=bins, alpha=.5, label='Chronic', density=True)
#plt.show()

In [ ]:
# same as abovve, but use kilosort instead so we don't need to reload waveforms
def compute_early_and_late_waveform_metrics_kilosort(unit_query, minutes=10):
    query = (unit_query * SpikeSorting.Unit * EphysRecording.ProbeSetting).proj()
    #times, amps, pos, fs = query.fetch('spike_times','spike_amplitudes','spike_positions','sampling_rate')
    beginning_amps, end_amps = [], []
    #for t, a, p, smplrate in tqdm(zip(times, amps, pos, fs)):
    for key in tqdm(query, desc='Fetching spike times and amplitudes'):
        t, a, p, smplrate = (SpikeSorting.Unit*EphysRecording.ProbeSetting & key).fetch1('spike_times','spike_amplitudes','spike_positions','sampling_rate')
        t = t / smplrate
        beginning_inds = np.where(t <= minutes * 60)[0]
        end_inds = np.where(t >= (t[-1] - minutes * 60))[0]
        beginning_amps.append(np.mean(a[beginning_inds]))
        end_amps.append(np.mean(a[end_inds]))
    beginning_amps = np.array(beginning_amps)
    end_amps = np.array(end_amps)
    return beginning_amps, end_amps 

#acute_early, acute_late = compute_early_and_late_waveform_metrics_kilosort(acute_p1f2)
#chronic_early, chronic_late = compute_early_and_late_waveform_metrics_kilosort(chronic_p1f2)
acute_early, acute_late = compute_early_and_late_waveform_metrics_kilosort(acute_passes_first)
chronic_early, chronic_late = compute_early_and_late_waveform_metrics_kilosort(chronic_passes_first)

In [ ]:
# normalize the amplitude drift by the amplitude of the unit
acute_abs_amp_drift = np.abs(acute_early - acute_late) / np.sum([np.abs(acute_early), np.abs(acute_late)], axis=0)
chronic_abs_amp_drift = np.abs(chronic_early - chronic_late) / np.sum([np.abs(chronic_early), np.abs(chronic_late)], axis=0)
acute_abs_amp_drift = acute_abs_amp_drift[~np.isnan(acute_abs_amp_drift) & ~np.isinf(acute_abs_amp_drift)]
chronic_abs_amp_drift = chronic_abs_amp_drift[~np.isnan(chronic_abs_amp_drift) & ~np.isinf(chronic_abs_amp_drift)]

x = np.random.normal(scale=.01, size=len(acute_abs_amp_drift))
x2 = np.random.normal(loc=.2, scale=.01, size=len(chronic_abs_amp_drift))

plt.figure()
#plt.scatter(x, acute_abs_amp_drift)
#plt.scatter(x2, chronic_abs_amp_drift)
bins = np.linspace(0, 20, 50)
bins = np.linspace(-10, 0, 50)
# convert to density

#plt.hist(acute_abs_amp_drift, bins=bins, alpha=.5, label='Acute')
#plt.hist(chronic_abs_amp_drift, bins=bins, alpha=.5, label='Chronic')
plt.hist(np.log(acute_abs_amp_drift), bins=bins, alpha=.5, label='Acute', density=True, color='k')
plt.hist(np.log(chronic_abs_amp_drift), bins=bins, alpha=.5, label='Chronic', density=True, color='red')
#plt.scatter(x, acute_abs_amp_drift, label='Acute', alpha=.5)
#plt.scatter(x2, chronic_abs_amp_drift, label='Chronic', alpha=.5)
#plt.yscale('log')
#plt.ylabel('Log Abs Amp Drift')
plt.xlabel('Log Amplitude Drift')
plt.ylabel('Density')
plt.legend()
if SAVEFIGS:
    plt.savefig(FIGPATH / 'log_amplitude_drift.pdf', dpi=800)

In [ ]:
# stats on this plot above
from operator import eq
from scipy.stats import ranksums, ttest_ind
stat, p_value = ranksums(np.log(acute_abs_amp_drift), np.log(chronic_abs_amp_drift))
stat2, p_value2 = ttest_ind(np.log(acute_abs_amp_drift), np.log(chronic_abs_amp_drift), equal_var=False)
p_value, p_value2

In [ ]:
ac = np.abs((UnitMetrics & acute_passes_first).fetch('depth_drift_start_to_end'))
cr = np.abs((UnitMetrics & chronic_passes_first).fetch('depth_drift_start_to_end'))

ac = ac[~np.isnan(ac)]
cr = cr[~np.isnan(cr)]
ac = ac[ac>0]
cr = cr[cr>0]

x = np.random.normal(scale=.01, size=len(ac))
x2 = np.random.normal(loc=.2, scale=.01, size=len(cr))

plt.figure()
plt.hist(np.log(ac), bins=30, alpha=.5, label='Acute', density=True, color='k')
plt.hist(np.log(cr), bins=30, alpha=.5, label='Chronic', density=True, color='red')
plt.legend()
plt.xlabel('Log Depth Drift')
plt.ylabel('Density')
if SAVEFIGS:
    plt.savefig(FIGPATH / 'log_depth_drift.pdf', dpi=800)

In [ ]:
from spks.viz import plot_footprints

sessions = EphysRecording.ProbeSetting.aggr(acute_p1f2).fetch(as_dict=True)
session_units = acute_p1f2 & sessions[9]
#sessions = EphysRecording.ProbeSetting.aggr(chronic_p1f2).fetch(as_dict=True)
#session_units = chronic_p1f2 & sessions[4]
_,_, chan_positions, early_waveforms, late_waveforms = compute_early_and_late_waveform_metrics(session_units)

In [ ]:
plt.figure()

unitinds = np.array([0,2,4,9,10,11,12,13,15,19,22])
g = (10,.1)
for u in unitinds:
    unit_key = (session_units).fetch('KEY')[u]
    print(u)
    print(unit_key)
    active_chans, times, pos, amps, gain, drift = (UnitMetrics*SpikeSorting.Unit*EphysRecording.ProbeSetting*ProbeConfiguration & unit_key).fetch1('active_electrodes',
                                                                                                                                            'spike_times',
                                                                                                                                            'spike_positions',
                                                                                                                                            'spike_amplitudes',
                                                                                                                                            'probe_gain',
                                                                                                                                            'depth_drift_start_to_end')
    pos = pos[:,1]
    times = times / 30000 / 60
    amps = (amps - np.mean(amps)) / np.std(amps)
    print(f'Unit {u} with drift {drift} um.')
    #amps = amps / gain
    # make a figure with side by side subplots
    fig, axs = plt.subplots(1,2, figsize=(8,4))
    plot_footprints(np.squeeze(early_waveforms[u][:,active_chans]), np.squeeze(chan_positions[u][active_chans,:]), gain=g, color='black', linewidth=2, alpha=.7,ax=axs[1])
    plot_footprints(np.squeeze(late_waveforms[u][:,active_chans]), np.squeeze(chan_positions[u][active_chans,:]), gain=g, color='red', linewidth=1.5, alpha=.7, ax=axs[1]) 
    _, scatter = plot_drift_raster(times, pos, amps, rasterized=True, ax=axs[0], cmap='bwr', clim=(-3,3))
    # remove top and right spines for axs[0]
    axs[0].spines[['top','right']].set_visible(False)
    axs[0].set_xlabel('Time (minutes)')
    axs[0].set_ylabel('Spike depth on shank (um)')
    # show colorbar on axs0
    cbar = plt.colorbar(scatter, location='right', fraction=.03, aspect=20,)
    cbar.ax.set_title('Spike \namplitude \n(z-score)', fontsize=8)
    cbar.ax.tick_params(labelsize=8)

    plt.tight_layout()
    if SAVEFIGS:
        plt.savefig(FIGPATH / f'acute_unit_{u}_footprint_drift.pdf', dpi=800)
    plt.show()

In [ ]:
# let's look at rasters for ones that fail the second

times, pos, amps = (SpikeSorting.Unit * session_units).fetch('spike_times','spike_positions','spike_amplitudes')

# restrict units
#times, pos, amps = times[unitinds], pos[unitinds], amps[unitinds]

times = times / 30000 / 60 
amps = np.hstack(amps)
pos = [p[:,1] for p in pos]
pos = np.hstack(pos)

plt.figure()
ids = [np.zeros_like(t)+np.random.uniform(0,1) for i,t in enumerate(times)]
ids = np.hstack(ids)
plot_drift_raster(np.hstack(times), pos, ids, cmap='tab20', rasterized=True)
plt.xlabel('Time (min)')
plt.ylabel('Position (um)')
if SAVEFIGS:
    plt.savefig(FIGPATH / 'acute_failed_units_example.pdf', dpi=800)

In [ ]:
chronic_insertion